# Correlation Matrix calcuation method 1:

This is the first method. Using the formulas from Lucy's paper we create the correlation matrix (more specifcaly, a low rank approximation of this correlation matrix) and use it to make predictions.

This method is non convex.

In [1]:
from tara_preprocessing import get_just_ecog_data,get_electrode_normalized_loc,car
from noah_production_funcs import single_patient_prediction_pure,create_lapaican_rbf,create_lapaican_knn,u_metric_display
from tara_preprocessing import remove_duplicates, hold_out, preprocessing,apply_car_function,clip_time_series
from tara_preprocessing import make_patient_correlation_matrix
from noah_production_funcs import create_u
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

### Loading in the data

Notice we are holding out from patient 0, removing electrodes 40 and 41.

For a run down of what all the preprocessing steps do, refer to the main README document for more.

In [ ]:
data_root = Path("/Users/noahwanless/Desktop/Spring2026/M467/faces_basic/data")
registered_dir = Path("../SuperEeg-M467-project/registered_outputs")
ecogs = get_just_ecog_data(registered_dir,data_root)
xyz = get_electrode_normalized_loc(registered_dir)
print('Downloaded data')
ecogs = clip_time_series(ecogs)
print("Time series clipped")
ecogs_no_dups,xyz_no_dups = remove_duplicates(ecogs,xyz)
print('Removed duplicate electrodes')
xyz_clea, cleane = preprocessing(ecogs_no_dups,xyz_no_dups,notch_size=.05)
print("Done Preprocessing")
cleaned_f,xyz_f,fake_pat_beginning,held_out_elcs = hold_out(xyz_clea,cleane,0,[40,41]) 
cleaned_f = apply_car_function(cleaned_f,0)
print("Done holding out electrodes")
patient_corr_mat = make_patient_correlation_matrix(xyz_f,cleaned_f)
print('Got Correlation Matrices, done!')

[PosixPath('../SuperEeg-M467-project/registered_outputs/aa_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ap_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ca_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/de_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/fp_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ha_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ja_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jm_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jt_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/mv_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/rn_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_output

In [4]:
U_det, loss = create_u(k=30,r=500,lamb=100,patient_corr_mat=patient_corr_mat,xyz_clean=xyz_f,training_steps=800,graph='rbf') #.0005

Optimizing U


100%|██████████| 800/800 [00:05<00:00, 142.26it/s]


In [3]:
pred,indices = single_patient_prediction_pure(0,cleaned_f,(U_det@U_det.T))

NameError: name 'U_det' is not defined